### Сверточные нейросети

In [1]:
import torch
import torch.nn as nn

### Первый способ создания последовательной модели

In [2]:
# input size => (batch_size, C, H, W)
# (кол-во каналов на вход, кол-во каналов на выход, (размер ядер сверточного фильтра))
model = nn.Sequential(
    nn.Conv2d(3, 32, (3,3)), #inp=(batch_size, 3, 28, 28), out=(batch_size, 32, 26, 26)
    nn.ReLU(),
    nn.Conv2d(32, 64, (3,3)) #inp=(batch_size, 32, 26, 26), out=(batch_size, 64, 64, 64)
)

model

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
)

Формулы для расчета выходных размеров тензора:
$$
H_{out} = \left\lfloor \frac{H_{in} + 2 \times padding[0] - 1 \times (kernel\_size[0] - 1) - 1}{stride[0]} + 1 \right\rfloor
$$

$$
W_{out} = \left\lfloor \frac{W_{in} + 2 \times padding[1] - 1 \times (kernel\_size[1] - 1) - 1}{stride[1]} + 1 \right\rfloor
$$

Частые комбинации размера ядра, паддинга и страйда:

In [3]:
'''
    kernel_size = (3,3), stride = (1,1), padding = (0,0)
    kernel_size = (3,3), stride = (1,1), padding = (1,1)
    kernel_size = (3,3), stride = (2,2), padding = (0,0)
'''

'\n    kernel_size = (3,3), stride = (1,1), padding = (0,0)\n    kernel_size = (3,3), stride = (1,1), padding = (1,1)\n    kernel_size = (3,3), stride = (2,2), padding = (0,0)\n'

In [4]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)
out = model(input)
out.shape

torch.Size([16, 64, 24, 24])

### Второй способ создания последовательной модели

In [5]:
model = nn.Sequential()
model.add_module('layer_1', nn.Conv2d(3, 32, (3,3))) #inp=(batch_size, 3, 28, 28), out=(batch_size, 32, 26, 26)
model.add_module('relu', nn.ReLU())
model.add_module('layer_2', nn.Conv2d(32, 64, (3,3))) #inp=(batch_size, 32, 26, 26), out=(batch_size, 64, 64, 64)

In [6]:
model

Sequential(
  (layer_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (layer_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
)

In [7]:
input = torch.rand([16, 3, 28, 28])
output = model(input)
output.shape

torch.Size([16, 64, 24, 24])

In [8]:
print(model.state_dict()['layer_1.weight'].shape)
print(model.state_dict()['layer_1.bias'].shape)
print(model.state_dict()['layer_2.weight'].shape)
# кол-во сверточных фильтров, кол-во ядер, размер ядра

torch.Size([32, 3, 3, 3])
torch.Size([32])
torch.Size([64, 32, 3, 3])


### Создание класса для модели нейросети

In [9]:
class MyModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_1 = nn.Conv2d(in_channels, 32, (3,3)) #inp=(batch,3,50,50), out=(batch,32,48,48)
        self.conv_2 = nn.Conv2d(32, out_channels, (3,3)) #inp=(batch,3,48,48), out=(batch,32,46,46)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu(x)
        out = self.conv_2(x)
        return out

model = MyModel(3, 64)
model

MyModel(
  (conv_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
)

In [10]:
input = torch.rand([16,3,50,50])
output = model(input)
output.shape

torch.Size([16, 64, 46, 46])

### Создание полноценной сверточной сети

In [11]:
tensor = torch.rand([2,1,3,3], dtype=torch.float32) #[batch, channels, H, W]
tensor

tensor([[[[0.6945, 0.8902, 0.7659],
          [0.9266, 0.8276, 0.9947],
          [0.6480, 0.2888, 0.8748]]],


        [[[0.2001, 0.5560, 0.2853],
          [0.4181, 0.5239, 0.8451],
          [0.8192, 0.4369, 0.4545]]]])

In [12]:
conv = nn.Conv2d(1,3,(3,3))(tensor)
print(conv)
print(conv.shape)

tensor([[[[0.1004]],

         [[0.1400]],

         [[0.3091]]],


        [[[0.0794]],

         [[0.0546]],

         [[0.1027]]]], grad_fn=<ConvolutionBackward0>)
torch.Size([2, 3, 1, 1])


In [13]:
tensor = tensor.reshape([2, -1])
print(tensor.shape)
print(tensor)

torch.Size([2, 9])
tensor([[0.6945, 0.8902, 0.7659, 0.9266, 0.8276, 0.9947, 0.6480, 0.2888, 0.8748],
        [0.2001, 0.5560, 0.2853, 0.4181, 0.5239, 0.8451, 0.8192, 0.4369, 0.4545]])


In [14]:
linear = nn.Linear(9, 10)(tensor)
print(linear.shape)
print(linear)

torch.Size([2, 10])
tensor([[ 0.3947,  0.4851, -0.0252,  0.3644,  0.5968,  0.4439, -0.9589, -0.4735,
          0.8665, -0.7061],
        [ 0.2760,  0.3227,  0.0303,  0.4227,  0.2354,  0.3389, -0.7826, -0.2983,
          0.5780, -0.3187]], grad_fn=<AddmmBackward0>)


Нулевая ось (размер батча = 2) во всех операциях не меняла своего значения во всех операциях

### Метод Flutten

В этом методе надо указывать какие оси надо объединить в одну

In [15]:
tensor = torch.rand([2,1,3,3], dtype=torch.float32)
tensor

tensor([[[[0.7255, 0.9397, 0.1791],
          [0.8678, 0.0306, 0.5170],
          [0.2514, 0.7028, 0.6735]]],


        [[[0.4228, 0.8753, 0.0046],
          [0.0665, 0.5775, 0.5170],
          [0.0058, 0.4554, 0.5087]]]])

In [16]:
flat = tensor.flatten(start_dim=1, end_dim=-1) #Оси с первой по последнюю (оси 1,3,3)
#flat = nn.Flatten()(tensor)

print(flat.shape)
print(flat)

torch.Size([2, 9])
tensor([[0.7255, 0.9397, 0.1791, 0.8678, 0.0306, 0.5170, 0.2514, 0.7028, 0.6735],
        [0.4228, 0.8753, 0.0046, 0.0665, 0.5775, 0.5170, 0.0058, 0.4554, 0.5087]])


In [17]:
class MyModel(nn.Module):
    def __init__(self, out):
        super().__init__()
        self.conv_1 = nn.Conv2d(3, 32, (3, 3)) # (batch_size, 3, 28, 28) => (batch_size, 32, 26, 26)
        self.conv_2 = nn.Conv2d(32, 64, (3, 3)) # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)

        self.flatten = nn.Flatten() # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.act = nn.ReLU()

        self.layer_1 = nn.Linear(64*24*24, 128)
        self.layer_2 = nn.Linear(128, out)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.act(x)
        x = self.conv_2(x)

        x = self.flatten(x)

        x = self.layer_1(x)
        x = self.act(x)
        out = self.layer_2(x)
        return out


model = MyModel(10)
model

MyModel(
  (conv_1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv_2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (act): ReLU()
  (layer_1): Linear(in_features=36864, out_features=128, bias=True)
  (layer_2): Linear(in_features=128, out_features=10, bias=True)
)

In [18]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)

out = model(input)
out.shape # (16, 10)

torch.Size([16, 10])

In [19]:
class MyModel(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False), # (batch_size, 3, 28, 28) => (batch_size, 32, 26, 26)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False), # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten() # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.fc = nn.Sequential(
            nn.Linear(64*24*24, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out


model = MyModel(3, 10)
model

MyModel(
  (conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc): Sequential(
    (0): Linear(in_features=36864, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [20]:
input = torch.rand([16, 3, 28, 28], dtype=torch.float32)

out = model(input)
out.shape # (16, 10)

torch.Size([16, 10])

### Модель для классификации MNIST

In [29]:
class MNISTModel(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False), # (batch_size, 3, 28, 28) => (batch_size, 32, 26, 26)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False), # (batch_size, 32, 26, 26) => (batch_size, 64, 24, 24)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten() # (batch_size, 64, 24, 24) => (batch_size, 64*24*24)

        self.fc = nn.Sequential(
            nn.Linear(64*24*24, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out

In [26]:
model_class = MNISTModel(1,10)

In [27]:
loss = nn.CrossEntropyLoss()
opt_class = torch.optim.Adam(model.parameters(), lr=0.001)

In [28]:
input = torch.rand([16,1,28,28], dtype=torch.float32)
output = model_class(input)
output.shape

torch.Size([16, 10])

### Модель для задачи регрессии

In [32]:
class RegModel(nn.Module):
    def __init__(self, in_channels, out):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, (3, 3), bias=False), # (batch_size, 3, 64, 64) => (batch_size, 32, 62, 62)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, (3, 3), bias=False), # (batch_size, 32, 62, 62) => (batch_size, 64, 60, 60)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.flatten = nn.Flatten() # (batch_size, 64, 60, 60) => (batch_size, 64*60*60)

        self.fc = nn.Sequential(
            nn.Linear(64*60*60, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, out)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        out = self.fc(x)
        return out


model_reg = RegModel(1, 2)

In [33]:
loss = nn.MSELoss()
opt_reg = torch.optim.Adam(model_reg.parameters(), lr=0.001)

In [35]:
input = torch.rand([16,1,64,64], dtype=torch.float32)
output = model_reg(input)
output.shape

torch.Size([16, 2])

### Вывод

* Сверточные слои заменяют часть полносвязных при работе с изображениями - они сохраняют пространственную структуру и резко сокращают число параметров по сравнению с Linear на сыром изображении
* Ключевая механика - расчёт выходных размеров после каждого Conv2d по формуле с kernel, padding, stride. без этого нельзя корректно собрать Flatten → Linear.
* BatchNorm2d после свёрток и Dropout перед выходным Linear-слоем - стандартная комбинация для стабилизации обучения и борьбы с переобучением в CNN.
* Одна и та же архитектура (блок conv+BatchNorm+ReLU → Flatten → fc) переиспользуется и для классфикации , и для регрессии. Меняется только последний слой и функция потерь.